# 03 — Análise Estatística

Objetivo: verificar formalmente, com correlação, teste de hipóteses e regressão,
os padrões observados na EDA sobre escolaridade, remuneração e setor econômico.

Como no restante do projeto, os resultados são interpretados como **associação**,
não causalidade.

In [4]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import pyarrow.parquet as pq

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

DATA_PROCESSED = Path("../data/processed")
FIGURES = Path("../figures")
arquivo = DATA_PROCESSED / "rais_sp_2025.parquet"

df = pq.ParquetFile(arquivo).read().to_pandas()
df.shape

(14948065, 16)

In [6]:
# Nomes originais -> nomes usados nas análises estatísticas.
# Baseado nas colunas efetivamente gravadas em data/processed/rais_sp_2025.parquet
# (ver 01_preparacao_dados.ipynb, colunas_processamento + colunas derivadas "Escolaridade"/"Setor").
rename_map = {
    "Vl Rem Média Nom": "remuneracao_media",
    "Vl Rem Dezembro Nom": "remuneracao_dezembro",
    "Escolaridade": "escolaridade",
    "Setor": "setor",
    "Idade": "idade",
    "Sexo - Código": "sexo",
    "Qtd Hora Contr": "horas_contratadas",
    "Tempo Emprego": "tempo_emprego",
    "Tipo Vínculo - Código": "tipo_vinculo",
    "Tamanho Estabelecimento - Código": "tamanho_estabelecimento",
    "CBO 2002 Ocupação - Código": "cbo_ocupacao",
}

df = df.rename(columns=rename_map)
df.columns.tolist()

['cbo_ocupacao',
 'CNAE 2.0 Classe - Código',
 'Ind Vínculo Ativo 31/12 - Código',
 'Escolaridade Após 2005 - Código',
 'horas_contratadas',
 'idade',
 'remuneracao_dezembro',
 'remuneracao_media',
 'sexo',
 'tamanho_estabelecimento',
 'tempo_emprego',
 'tipo_vinculo',
 'Ind Trabalho Intermitente - Código',
 'escolaridade',
 'CNAE_Divisao',
 'setor']

In [7]:
# Mesma ordem/rótulos definidos no 01 (mapa_escolaridade) e usados no 02.
# Importante: são os textos EXATOS já gravados na coluna "escolaridade" do parquet.
ordem_escolaridade = [
    "Analfabeto",
    "Até 5º ano incompleto",
    "5º ano completo do Fundamental",
    "6º ao 9º ano do Fundamental",
    "Fundamental completo",
    "Médio incompleto",
    "Médio completo",
    "Superior incompleto",
    "Superior completo",
    "Mestrado",
    "Doutorado",
]

df["escolaridade"] = pd.Categorical(
    df["escolaridade"], categories=ordem_escolaridade, ordered=True
)

# Checagem: nenhum valor deve virar NaN nesse passo (senão, algum rótulo não bateu)
print("Escolaridade não reconhecida:", df["escolaridade"].isna().sum())

# rank ordinal — usado SÓ no Spearman, nunca como variável numérica na regressão
df["escolaridade_rank"] = df["escolaridade"].cat.codes

df["log_remuneracao"] = np.log(df["remuneracao_media"])

df[["escolaridade", "escolaridade_rank", "remuneracao_media", "log_remuneracao", "setor"]].head()

Escolaridade não reconhecida: 0


,escolaridade,escolaridade_rank,remuneracao_media,log_remuneracao,setor
0,Médio completo,6,6560.88,8.788880,Comércio
1,6º ao 9º ano do Fundamental,3,2872.43,7.962914,Serviços
2,Superior completo,8,4239.73,8.352255,Comércio
3,Médio completo,6,3995.73,8.292982,Serviços
4,Superior incompleto,7,2580.32,7.855669,Serviços


## Tratamento de valores ausentes/sentinela nas variáveis de controle

`escolaridade`, `setor` e `remuneracao_media` já foram tratados no 01 (são as
variáveis centrais da população analítica). As variáveis de controle
(`sexo`, `tipo_vinculo`, `tamanho_estabelecimento`, `idade`, `horas_contratadas`,
`tempo_emprego`) ainda não passaram por esse tratamento — seguindo o mesmo
critério documentado no 01 ("tratar códigos classificados como ignorados (-1)
como valores ausentes"), fazemos isso agora, só para as variáveis que entram
no modelo.